In [ ]:
import ast
import re
import torch
from PIL import Image


def parse_validation_output(output_text):
    pattern = r"\[\s*[1-4]\s*,\s*[1-4]\s*,\s*[1-4]\s*,\s*[1-4]\s*\]"
    match = re.search(pattern, output_text)

    if match is None:
        return None

    try:
        result = ast.literal_eval(match.group())
    except (ValueError, SyntaxError, TypeError):
        return None

    if isinstance(result, list) and len(result) == 4 and sorted(result) == [1, 2, 3, 4]:
        return [int(value) for value in result]

    return None


def predict_validation_example(example):
    images = []
    for image_path in example["image_paths"]:
        with Image.open(image_path) as image:
            images.append(image.convert("RGB").copy())

    user_content = []
    for image_label in example["image_labels"]:
        user_content.append({"type": "text", "text": f"\n{image_label}:"})
        user_content.append({"type": "image"})

    user_content.append({
        "type": "text",
        "text": "\n\n" + example["instruction"]
    })

    messages = [{"role": "user", "content": user_content}]
    text = processor.apply_chat_template(
        messages,
        tokenize=False,
        add_generation_prompt=True
    )

    inputs = processor(
        text=[text],
        images=images,
        padding=False,
        return_tensors="pt"
    ).to(model.device)

    with torch.no_grad():
        generated_ids = model.generate(
            **inputs,
            max_new_tokens=32,
            do_sample=False
        )

    generated_ids_trimmed = [
        out_ids[len(in_ids):]
        for in_ids, out_ids in zip(inputs.input_ids, generated_ids)
    ]

    raw_output = processor.batch_decode(
        generated_ids_trimmed,
        skip_special_tokens=True,
        clean_up_tokenization_spaces=False
    )[0]

    return raw_output, parse_validation_output(raw_output)

import pandas as pd
from tqdm.auto import tqdm

model.eval()

# 평가 중에는 생성 속도를 위해 cache 사용
model.config.use_cache = True

N_EVAL = len(validation_dataset)

validation_results = []

for index in tqdm(range(N_EVAL), desc="Validation"):
    example = validation_dataset[index]

    raw_output, prediction = predict_validation_example(
        example
    )

    answer = ast.literal_eval(example["target"])
    answer = [int(value) for value in answer]

    validation_results.append({
        "Id": example["Id"],
        "Raw_output": raw_output,
        "Prediction": (
            str(prediction)
            if prediction is not None
            else None
        ),
        "Answer": str(answer),
        "Correct": prediction == answer
    })

validation_result_df = pd.DataFrame(validation_results)

display(validation_result_df)

In [ ]:
exact_match_accuracy = (
    validation_result_df["Correct"].mean()
)

parse_failure_count = (
    validation_result_df["Prediction"].isna().sum()
)

print(
    f"Exact-match accuracy: "
    f"{exact_match_accuracy:.2%}"
)

print(
    f"맞힌 개수: "
    f"{validation_result_df['Correct'].sum()}"
    f"/{len(validation_result_df)}"
)

print(
    f"파싱 실패 개수: "
    f"{parse_failure_count}"
)

In [ ]:
exact_match_accuracy = (
    validation_result_df["Correct"].mean()
)

parse_failure_count = (
    validation_result_df["Prediction"].isna().sum()
)

print(
    f"Exact-match accuracy: "
    f"{exact_match_accuracy:.2%}"
)

print(
    f"맞힌 개수: "
    f"{validation_result_df['Correct'].sum()}"
    f"/{len(validation_result_df)}"
)

print(
    f"파싱 실패 개수: "
    f"{parse_failure_count}"
)

In [ ]:
identity_answer = [1, 2, 3, 4]

identity_correct = validation_result_df["Answer"].apply(
    lambda answer_text:
    ast.literal_eval(answer_text) == identity_answer
)

identity_baseline_accuracy = identity_correct.mean()

print(
    f"항상 [1,2,3,4] baseline: "
    f"{identity_baseline_accuracy:.2%}"
)

print(
    f"학습 모델 exact-match: "
    f"{exact_match_accuracy:.2%}"
)

In [ ]:
identity_answer = [1, 2, 3, 4]

identity_correct = validation_result_df["Answer"].apply(
    lambda answer_text:
    ast.literal_eval(answer_text) == identity_answer
)

identity_baseline_accuracy = identity_correct.mean()

print(
    f"항상 [1,2,3,4] baseline: "
    f"{identity_baseline_accuracy:.2%}"
)

print(
    f"학습 모델 exact-match: "
    f"{exact_match_accuracy:.2%}"
)

In [ ]:
frame_columns = [
    "Input_1",
    "Input_2",
    "Input_3",
    "Input_4"
]

train_ids = set(training_df["Id"].astype(str))
validation_ids = set(validation_df["Id"].astype(str))

print(
    "중복 Id:",
    len(train_ids & validation_ids)
)

def frame_signature(row):
    return tuple(
        sorted(str(row[column]) for column in frame_columns)
    )

train_signatures = set(
    training_df.apply(frame_signature, axis=1)
)

validation_signatures = set(
    validation_df.apply(frame_signature, axis=1)
)

print(
    "동일 프레임 묶음:",
    len(train_signatures & validation_signatures)
)

In [ ]:
frame_columns = [
    "Input_1",
    "Input_2",
    "Input_3",
    "Input_4"
]

train_ids = set(training_df["Id"].astype(str))
validation_ids = set(validation_df["Id"].astype(str))

print(
    "중복 Id:",
    len(train_ids & validation_ids)
)

def frame_signature(row):
    return tuple(
        sorted(str(row[column]) for column in frame_columns)
    )

train_signatures = set(
    training_df.apply(frame_signature, axis=1)
)

validation_signatures = set(
    validation_df.apply(frame_signature, axis=1)
)

print(
    "동일 프레임 묶음:",
    len(train_signatures & validation_signatures)
)

In [ ]:
from PIL import Image


def find_last_subsequence(sequence, pattern):
    """sequence에서 pattern이 마지막으로 등장하는 시작 위치."""
    max_start = len(sequence) - len(pattern)

    for start in range(max_start, -1, -1):
        if sequence[start:start + len(pattern)] == pattern:
            return start

    return -1


class QwenFrameOrderCollator:
    def __init__(self, processor):
        self.processor = processor

        self.assistant_prefix_ids = (
            processor.tokenizer.encode(
                "<|im_start|>assistant\n",
                add_special_tokens=False
            )
        )

    @staticmethod
    def load_image(path):
        with Image.open(path) as image:
            return image.convert("RGB").copy()

    def __call__(self, examples):
        # T4 메모리를 고려해 batch size 1만 사용
        if len(examples) != 1:
            raise ValueError(
                "현재 Collator는 batch size 1 전용입니다."
            )

        example = examples[0]

        images = [
            self.load_image(path)
            for path in example["image_paths"]
        ]

        user_content = []

        for image_label in example["image_labels"]:
            user_content.append({
                "type": "text",
                "text": f"\n{image_label}:"
            })

            user_content.append({
                "type": "image"
            })

        user_content.append({
            "type": "text",
            "text": "\n\n" + example["instruction"]
        })

        full_messages = [
            {
                "role": "user",
                "content": user_content
            },
            {
                "role": "assistant",
                "content": example["target"]
            }
        ]

        full_text = self.processor.apply_chat_template(
            full_messages,
            tokenize=False,
            add_generation_prompt=False
        )

        model_inputs = self.processor(
            text=[full_text],
            images=images,
            padding=False,
            return_tensors="pt"
        )

        input_id_list = (
            model_inputs["input_ids"][0]
            .tolist()
        )

        assistant_position = find_last_subsequence(
            input_id_list,
            self.assistant_prefix_ids
        )

        if assistant_position >= 0:
            answer_start = (
                assistant_position
                + len(self.assistant_prefix_ids)
            )

        else:
            # 템플릿 버전에 따른 안전한 fallback
            prompt_messages = [{
                "role": "user",
                "content": user_content
            }]

            prompt_text = (
                self.processor.apply_chat_template(
                    prompt_messages,
                    tokenize=False,
                    add_generation_prompt=True
                )
            )

            prompt_inputs = self.processor(
                text=[prompt_text],
                images=images,
                padding=False,
                return_tensors="pt"
            )

            answer_start = (
                prompt_inputs["input_ids"]
                .shape[1]
            )

        labels = model_inputs["input_ids"].clone()

        # 이미지·질문·지시문에는 loss를 주지 않음
        labels[:, :answer_start] = -100

        if "attention_mask" in model_inputs:
            labels[
                model_inputs["attention_mask"] == 0
            ] = -100

        model_inputs["labels"] = labels

        return model_inputs


data_collator = QwenFrameOrderCollator(
    processor
)

In [ ]:
import ast
import pandas as pd

# Id 자료형 통일
validation_result_df = validation_result_df.copy()
validation_df = validation_df.copy()

validation_result_df["Id"] = validation_result_df["Id"].astype(str)
validation_df["Id"] = validation_df["Id"].astype(str)


# No_ordering을 확실한 bool로 변환
def to_bool(value):
    if isinstance(value, bool):
        return value

    value = str(value).strip().lower()

    if value in {"true", "1", "yes"}:
        return True
    if value in {"false", "0", "no"}:
        return False

    raise ValueError(f"알 수 없는 No_ordering 값: {value}")


validation_df["No_ordering"] = (
    validation_df["No_ordering"].apply(to_bool)
)


# 예측 결과와 원본 정보 합치기
result_with_group = validation_result_df.merge(
    validation_df[["Id", "No_ordering"]],
    on="Id",
    how="left",
    validate="one_to_one"
)


def parse_order(value):
    if value is None or pd.isna(value):
        return None

    if isinstance(value, list):
        parsed = value
    else:
        try:
            parsed = ast.literal_eval(str(value))
        except (ValueError, SyntaxError):
            return None

    if (
        not isinstance(parsed, list)
        or len(parsed) != 4
    ):
        return None

    try:
        return [int(x) for x in parsed]
    except (TypeError, ValueError):
        return None


def calculate_row_metrics(row):
    prediction = parse_order(row["Prediction"])
    answer = parse_order(row["Answer"])

    if prediction is None or answer is None:
        return pd.Series({
            "Position_correct": 0,
            "Position_total": 4,
            "Exact_correct": 0,
            "Parse_failed": 1
        })

    position_correct = sum(
        pred == target
        for pred, target in zip(prediction, answer)
    )

    return pd.Series({
        "Position_correct": position_correct,
        "Position_total": 4,
        "Exact_correct": int(prediction == answer),
        "Parse_failed": 0
    })


metrics = result_with_group.apply(
    calculate_row_metrics,
    axis=1
)

result_with_group = pd.concat(
    [result_with_group, metrics],
    axis=1
)


group_accuracy = (
    result_with_group
    .groupby("No_ordering")
    .agg(
        sample_count=("Id", "count"),
        correct_positions=("Position_correct", "sum"),
        total_positions=("Position_total", "sum"),
        exact_correct=("Exact_correct", "sum"),
        parse_failures=("Parse_failed", "sum")
    )
)

group_accuracy["position_accuracy"] = (
    group_accuracy["correct_positions"]
    / group_accuracy["total_positions"]
)

group_accuracy["exact_match_accuracy"] = (
    group_accuracy["exact_correct"]
    / group_accuracy["sample_count"]
)

print(group_accuracy)